# Options from Zero · Episode 4: Forwards and put-call parity

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **The running trade.** An illustrative share priced at \$100 that pays no dividends, a one-year call option on it with a strike of \$105, 20% volatility and a 3.60% interest rate (continuously compounded). All made up for teaching; it is not market data.
>
> Educational material only, not financial advice. Options are risky: a seller can lose far more than the premium received.

1. Setup · 2. The forward price · 3. Two portfolios, one payoff · 4. Put-call parity · 5. Why the put costs more · 6. Catching a mispriced put · 7. Check against QuantLib · 8. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data files this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import QuantLib as ql

out = {"meta": {
    "series": "Options from Zero",
    "episode": 4,
    "quantlib_version": ql.__version__,
    "quotes_label": "illustrative (made up for teaching)",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}
S, K, T, r, vol = 100.0, 105.0, 1.0, 0.036, 0.20   # the running trade

The option formulas used throughout the series (`episodes/_shared/options.py`):

In [ ]:
# Option formulas for Options from Zero, written out by hand so every step can be read. Copied verbatim into
# each notebook by make_notebook.py, so every notebook runs on its own; the notebooks check these against
# QuantLib. Needs: math, numpy as np.
#   Running trade (Part 1): an illustrative share at $100, no dividends, 20% volatility, a one-year call struck
#   at $105, interest 3.60% a year continuously compounded. Made up for teaching.

def norm_cdf(x):
    """N(x): the chance a standard normal variable is below x."""
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def call_payoff(s, k):
    return np.maximum(np.asarray(s, dtype=float) - k, 0.0)

def put_payoff(s, k):
    return np.maximum(k - np.asarray(s, dtype=float), 0.0)

def black_scholes(s, k, t, r, vol, call=True):
    """Black-Scholes price of a European option on a share with no dividends (r continuously compounded)."""
    d1 = (math.log(s / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    d2 = d1 - vol * math.sqrt(t)
    if call:
        return s * norm_cdf(d1) - k * math.exp(-r * t) * norm_cdf(d2)
    return k * math.exp(-r * t) * norm_cdf(-d2) - s * norm_cdf(-d1)

def one_step(s, up, down, k, t, r, call=True):
    """One-step tree: replicate the option with delta shares and cash. Returns the pieces and the price."""
    pay = call_payoff if call else put_payoff
    v_up, v_down = float(pay(up, k)), float(pay(down, k))
    delta = (v_up - v_down) / (up - down)
    cash = (v_down - delta * down) * math.exp(-r * t)   # negative: borrowed today
    q = (s * math.exp(r * t) - down) / (up - down)       # risk-neutral probability of the up move
    return {"v_up": v_up, "v_down": v_down, "delta": delta, "cash": cash, "price": delta * s + cash, "q": q}

def binomial_tree(s, k, t, r, vol, steps, call=True, american=False, quantlib_p=False):
    """Cox-Ross-Rubinstein tree: u = exp(vol sqrt(dt)), d = 1/u, worked backward from expiry.
    quantlib_p=True uses QuantLib's up-probability 1/2 + 1/2 (r - vol^2/2) sqrt(dt) / vol instead of the textbook
    (e^{r dt} - d) / (u - d); both give the same price in the limit."""
    dt = t / steps
    u = math.exp(vol * math.sqrt(dt)); d = 1 / u
    p = 0.5 + 0.5 * (r - vol ** 2 / 2) * math.sqrt(dt) / vol if quantlib_p else (math.exp(r * dt) - d) / (u - d)
    pay = call_payoff if call else put_payoff
    prices = s * d ** np.arange(steps, -1, -1) * u ** np.arange(steps + 1)
    values = pay(prices, k)
    for _ in range(steps):
        prices = prices[1:] / u
        values = math.exp(-r * dt) * (p * values[1:] + (1 - p) * values[:-1])
        if american:
            values = np.maximum(values, pay(prices, k))
    return float(values[0])

## 2. The forward price

Agree today to buy the share in a year. What price is fair? Borrow \$100, buy the share now, hold it: in a year you own the share and owe $100 e^{rT}$. Any other forward price hands someone a free profit.

In [ ]:
F = S * math.exp(r * T)
pv_k = K * math.exp(-r * T)
out["forward"] = {"spot": S, "forward": F, "borrow": S, "repay": F, "pv_strike": pv_k, "strike": K}
out["forward"]

## 3. Two portfolios, one payoff

Buy the call and sell the put, both struck at \$105. At expiry: above the strike the call pays $S_T - K$; below it you owe the put's $K - S_T$. Either way you end up with exactly $S_T - K$, the same as a forward agreement to buy at \$105.

In [ ]:
grid = np.arange(60.0, 150.5, 1.0)
call_pay, put_pay = call_payoff(grid, K), put_payoff(grid, K)
combo = call_pay - put_pay
assert np.allclose(combo, grid - K)
out["chart"] = {"x": grid.tolist(), "long_call": call_pay.tolist(), "short_put": (-put_pay).tolist(),
                "together": combo.tolist(), "ticks": [{"x": float(x), "label": f"${x:.0f}"} for x in range(60, 151, 15)]}

## 4. Put-call parity

Same payoff, same price today: call − put = share − strike × $e^{-rT}$. No model, no volatility, no opinions: just no free money.

In [ ]:
C = black_scholes(S, K, T, r, vol, True)
P = black_scholes(S, K, T, r, vol, False)
out["parity"] = {"call": C, "put": P, "lhs": C - P, "rhs": S - pv_k, "lhs_abs": abs(C - P), "gap": P - C,
                 "residual": (C - P) - (S - pv_k)}
# parity holds for every strike, whatever the volatility
checks = [black_scholes(S, k, T, r, v, True) - black_scholes(S, k, T, r, v, False) - (S - k * math.exp(-r * T))
          for k in (80, 100, 105, 130) for v in (0.1, 0.2, 0.4)]
assert max(abs(c) for c in checks) < 1e-12
out["parity"]

## 5. Why the put costs more

Parity says the call and the put cost the same exactly when the strike equals the forward, \$103.67. Our strike, \$105, is above the forward, so the put is dearer: it's already "in the money" against the forward. Nothing to do with fear.

In [ ]:
rows = []
for k in (100.0, F, 105.0):
    rows.append({"strike": k, "call": black_scholes(S, k, T, r, vol, True), "put": black_scholes(S, k, T, r, vol, False)})
out["by_strike"] = {"rows": rows, "at_forward_call": rows[1]["call"], "at_forward_put": rows[1]["put"]}
assert abs(rows[1]["call"] - rows[1]["put"]) < 1e-12
pd.DataFrame(rows)

## 6. Catching a mispriced put

Suppose someone quotes the \$105 put 50 cents too cheap, rounded to the cent. Buy the put and the share, sell the call, borrow the present value of \$105. At expiry, share + put − call is worth exactly \$105 whatever happens, which repays the loan. The money left over today is a sure profit.

In [ ]:
quote = round(P - 0.50, 2)
today_total = C - quote - S + pv_k                   # sell call, buy put, buy share, borrow PV(K)
rows = [{"when": "today", "call": C, "put": -quote, "share": -S, "loan": pv_k, "total": today_total}]
for s_T in (80.0, 105.0, 130.0):
    rows.append({"when": f"share at ${s_T:.0f}", "call": -float(call_payoff(s_T, K)), "put": float(put_payoff(s_T, K)),
                 "share": s_T, "loan": -K, "total": -float(call_payoff(s_T, K)) + float(put_payoff(s_T, K)) + s_T - K})
assert all(abs(x["total"]) < 1e-12 for x in rows[1:])
out["arb"] = {"quote": quote, "fair": P, "profit": today_total, "rows": rows, "loan": pv_k}
pd.DataFrame(rows)

## 7. Check against QuantLib

In [ ]:
today = ql.Date(29, 9, 2026)
ql.Settings.instance().evaluationDate = today
dc = ql.Actual365Fixed()
flat = lambda x: ql.YieldTermStructureHandle(ql.FlatForward(today, x, dc, ql.Continuous))
process = ql.BlackScholesMertonProcess(
    ql.QuoteHandle(ql.SimpleQuote(S)), flat(0.0), flat(r),
    ql.BlackVolTermStructureHandle(ql.BlackConstantVol(today, ql.NullCalendar(), vol, dc)))
expiry = today + 365   # exactly one year on ACT/365F
prices = {}
for name, kind in (("call", ql.Option.Call), ("put", ql.Option.Put)):
    opt = ql.VanillaOption(ql.PlainVanillaPayoff(kind, K), ql.EuropeanExercise(expiry))
    opt.setPricingEngine(ql.AnalyticEuropeanEngine(process))
    prices[name] = opt.NPV()
fwd_ql = S / flat(r).discount(expiry)
out["quantlib"] = {**prices, "forward": fwd_ql, "parity_residual": prices["call"] - prices["put"] - (S - K * flat(r).discount(expiry))}
assert abs(out["quantlib"]["parity_residual"]) < 1e-12 and abs(fwd_ql - F) < 1e-10
out["quantlib"]

## 8. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")